In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

In [2]:
import os
from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())



PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
os.environ['DATABASE_URL'] = pg.get_uri()
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [4]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = '/content/yellow_tripdata_2024-03.parquet'
urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(parquet_path, columns=columns).sample(n=500_000, random_state=42)

# Basic cleaning before loading to PostgreSQL
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine)


,n_rows
0,480035


In [5]:
%%sql
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

Done.
5420 rows affected.
Done.
10 rows affected.


pickup_zone_id,n_days
1,14
3,12
4,31
6,3
7,31
9,9
10,31
11,14
12,31
13,31


Завдання 2

Lag features

In [6]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-03-04,1,None,None,None
1,2024-03-08,1,1,None,None
1,2024-03-10,2,1,None,None
1,2024-03-11,1,2,None,None
1,2024-03-14,1,1,None,None
1,2024-03-18,1,1,None,None
1,2024-03-21,2,1,None,None
1,2024-03-22,1,2,1,None
1,2024-03-23,1,1,1,None
1,2024-03-24,1,1,2,None


Rolling 7-day average fare

In [7]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-03-04,40.00,40.0000000000000000,1
1,2024-03-08,105.00,72.5000000000000000,2
1,2024-03-10,23.00,56.0000000000000000,3
1,2024-03-11,3.00,43.6666666666666667,3
1,2024-03-14,87.40,54.6000000000000000,4
1,2024-03-18,128.30,107.8500000000000000,2
1,2024-03-21,86.55,107.4250000000000000,2
1,2024-03-22,82.80,99.2166666666666667,3
1,2024-03-23,105.00,100.6625000000000000,4
1,2024-03-24,35.00,87.5300000000000000,5


Rolling 30-day fare volatility

In [8]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-03-04,40.00,None,1
1,2024-03-08,105.00,45.9619407771255891,2
1,2024-03-10,23.00,43.2781700167648031,3
1,2024-03-11,3.00,44.1691823182936755,4
1,2024-03-14,87.40,43.1498783312305528,5
1,2024-03-18,128.30,49.6786372598927073,6
1,2024-03-21,86.55,46.1130352916561307,7
1,2024-03-22,82.80,43.0289639785973533,8
1,2024-03-23,105.00,41.9527561669075563,9
1,2024-03-24,35.00,41.3801243889435939,10


Expanding cumulative trips

In [9]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-03-04,1,1,1.00000000000000000000
1,2024-03-08,1,2,1.00000000000000000000
1,2024-03-10,2,4,1.3333333333333333
1,2024-03-11,1,5,1.2500000000000000
1,2024-03-14,1,6,1.2000000000000000
1,2024-03-18,1,7,1.1666666666666667
1,2024-03-21,2,9,1.2857142857142857
1,2024-03-22,1,10,1.2500000000000000
1,2024-03-23,1,11,1.2222222222222222
1,2024-03-24,1,12,1.2000000000000000


Combined feature view

In [10]:
%%sql
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
20 rows affected.


pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
4,2024-03-31,63,16.93,181.13,65,68,25,17.1371428571428571,2.4893482505755755,995,32.0967741935483871
7,2024-03-31,16,18.03,48.15,12,12,13,19.7828571428571429,4.5389829358518218,306,9.8709677419354839
10,2024-03-31,5,42.87,49.98,1,6,5,50.6957142857142857,14.1980685751396417,163,5.2580645161290323
12,2024-03-31,6,33.22,45.02,11,5,4,25.9100000000000000,5.8765721202974674,152,4.9032258064516129
13,2024-03-31,45,22.31,201.07,76,56,90,21.6042857142857143,2.2001391021854771,2786,89.8709677419354839
17,2024-03-31,4,36.70,33.67,5,4,3,25.8071428571428571,9.6446765734728789,132,4.2580645161290323
24,2024-03-31,33,15.17,88.44,31,34,46,15.7814285714285714,2.0405686890454444,1204,38.8387096774193548
33,2024-03-31,5,33.38,35.79,7,12,9,28.7928571428571429,5.3859440308499564,294,9.4838709677419355
35,2024-03-31,4,42.25,41.94,2,5,11,31.9700000000000000,5.8855998922755549,125,4.0322580645161290
39,2024-03-31,6,27.85,39.32,4,2,2,31.2442857142857143,10.3485443698479954,128,4.1290322580645161


Завдання 3

In [11]:
%%sql
DROP TABLE IF EXISTS t7_hw_events CASCADE;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
50000 rows affected.
Done.
5 rows affected.


event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


Query 1. Containment @>

In [12]:
%%sql
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


Query 2. Key existence ? та ?|

In [13]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [14]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


Query 3. JSONPath

In [15]:
%%sql
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
15 rows affected.


event_id,recommended_pid
4,5
4,6
4,7
9,10
9,11
9,12
14,15
14,16
14,17
19,20


In [16]:
%%sql
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


Завдання 4

Containment query без GIN-індексу

In [17]:
%%sql
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
Done.
Done.
10 rows affected.


QUERY PLAN
Aggregate (cost=1755.75..1755.76 rows=1 width=8) (actual time=56.064..56.066 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=2300 width=0) (actual time=0.014..55.754 rows=3334 loops=1)
"Filter: (payload @> jsonb_build_object('payment_method', 'card'))"
Rows Removed by Filter: 46666
Buffers: shared hit=1000
Planning:
Buffers: shared hit=7
Planning Time: 0.234 ms
Execution Time: 56.092 ms


GIN-індекс для containment / JSONPath workload

In [18]:
%%sql
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
1 rows affected.


gin_index_size
1144 kB


Той самий containment query після GIN

In [19]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
13 rows affected.


QUERY PLAN
Aggregate (cost=1072.61..1072.62 rows=1 width=8) (actual time=14.343..14.345 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=26.62..1066.04 rows=2628 width=0) (actual time=0.989..13.876 rows=3334 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..25.97 rows=2628 width=0) (actual time=0.691..0.692 rows=3334 loops=1)
"Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Buffers: shared hit=4
Planning:


In [20]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
13 rows affected.


QUERY PLAN
Aggregate (cost=1062.23..1062.24 rows=1 width=8) (actual time=6.503..6.505 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=23.10..1057.34 rows=1957 width=0) (actual time=0.735..6.278 rows=2500 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..22.61 rows=1957 width=0) (actual time=0.488..0.489 rows=2500 loops=1)
"Index Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Buffers: shared hit=4
Planning:


Коли GIN не допомагає: path-specific text equality

In [21]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
8 rows affected.


QUERY PLAN
Aggregate (cost=1750.62..1750.63 rows=1 width=8) (actual time=14.706..14.708 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=250 width=0) (actual time=0.022..14.079 rows=8333 loops=1)
Filter: ((payload ->> 'device'::text) = 'mobile'::text)
Rows Removed by Filter: 41667
Buffers: shared hit=1000
Planning Time: 0.095 ms
Execution Time: 14.741 ms


expression index

In [22]:
%%sql
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
13 rows affected.


QUERY PLAN
Aggregate (cost=1245.48..1245.49 rows=1 width=8) (actual time=3.452..3.453 rows=1 loops=1)
Buffers: shared hit=1000 read=9
-> Bitmap Heap Scan on t7_hw_events (cost=97.72..1224.37 rows=8443 width=0) (actual time=0.583..2.898 rows=8333 loops=1)
Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Heap Blocks: exact=1000
Buffers: shared hit=1000 read=9
-> Bitmap Index Scan on idx_t7_events_device_btree (cost=0.00..95.61 rows=8443 width=0) (actual time=0.409..0.409 rows=8333 loops=1)
Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Buffers: shared read=9
Planning:


Завдання 5

Step 1. Нормалізована таблиця для checkout-подій

In [23]:
%%sql
DROP TABLE IF EXISTS t7_hw_checkouts_norm CASCADE;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
10000 rows affected.
Done.
Done.
1 rows affected.


n_checkouts
10000


Step 2. Один запит двома способами

JSONB-версія:

In [24]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
18 rows affected.


QUERY PLAN
Limit (cost=1958.53..1958.54 rows=3 width=72) (actual time=17.848..17.850 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1958.53..1972.56 rows=5610 width=72) (actual time=17.846..17.847 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1801.87..1886.02 rows=5610 width=72) (actual time=17.797..17.834 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


Normalized-версія:

In [25]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=9.237..9.239 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=9.235..9.237 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=9.221..9.224 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


Step 3. Bonus: MongoDB-equivalent як pseudocode



```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

Порівняння
- PostgreSQL JSONB зручний, коли поряд із гнучкими документами потрібні SQL, JOIN з нормалізованими таблицями (наприклад, users чи products), транзакції та constraints, і все це в одній базі
- MongoDB aggregation pipeline природний для document-first навантажень: коли дані від початку живуть як документи, схема часто змінюється, а основні запити - вибірка й агрегація всередині однієї колекції
- High-volume raw clickstream часто краще зберігати в document store або data lake, де дешевий запис і горизонтальне масштабування
- Curated ML features зручніше тримати в typed SQL-таблицях: типи й constraints захищають від сміття, запити простіші, а агрегації дешевші.

Reflection

1. Що змінилось після GIN-індексу?

До індексу PostgreSQL використав Seq Scan, тобто переглянув усю таблицю: 1000 сторінок. З 50 000 рядків він відкинув 46 666, щоб знайти 3 334 потрібні. Total cost був 1755.76, а запит виконувався 56.1 ms.

Після GIN-індексу план змінився на Bitmap Index Scan + Bitmap Heap Scan. Cost став 1072.62, а час 14.4 ms, тобто приблизно в 4 рази швидше. Запит з referrer = google теж використав індекс і виконався за 6.6 ms.

Я помітила, що Heap Blocks все одно 1000, тобто таблиця все одно прочитана вся. Думаю, це тому, що потрібні рядки є майже на кожній сторінці. Але PostgreSQL перевіряв умову тільки для рядків, які знайшов індекс, тому вийшло швидше.

2. Чому GIN не допоміг для device = 'mobile'?

GIN працює з @> і JSONPath, а payload ->> 'device' = 'mobile' - це звичайне порівняння тексту, і GIN тут не використовується. Тому навіть з GIN був Seq Scan (14.7 ms). Ще PostgreSQL очікував 250 рядків, а насправді їх було 8 333.

Коли я створила B-tree індекс саме на payload ->> 'device', з'явився Bitmap Index Scan, і час став 3.5 ms. Оцінка рядків теж стала майже точною: 8 443.

3. Що стає гіршим через GIN?

Читати стає швидше, а записувати повільніше: при кожному INSERT чи UPDATE треба оновити ще й індекс, а GIN зберігає всі ключі і значення з документа. Ще індекс займає місце, у мене 1144 kB. Швидкість запису я окремо не перевіряла.

4. Коли JSONB, а коли звичайна таблиця?

Я порівняла однаковий запит. У JSONB PostgreSQL переглянув усі 50 000 рядків (40 000 відкинув), прочитав 1000 сторінок, час 17.9 ms. У звичайній таблиці лише 10 000 рядків, 94 сторінки, 9.3 ms. Ще в JSONB він думав, що буде 5 610 груп, а їх лише 3. У звичайній таблиці вгадав точно.

Тому сирі події я б тримала в JSONB, бо в різних подій різні поля. А готові ознаки для ML - у звичайній таблиці з типами і перевірками.

5. Як додати новий тип події refund?

У JSONB таблицю міняти не треба: просто додаємо рядки з event_type = 'refund' і своїм payload. Але тоді ніхто не перевірить, що поля правильні.

У звичайному варіанті треба створити нову таблицю для refund з колонками і перевірками, додати індекс і перенести туди дані.